## 0. Setup

In [3]:
# Uncomment for Google Colab
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_PATH = '/content/drive/MyDrive/drug_response_prediction'
# import os
# os.chdir(PROJECT_PATH)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path
import warnings
import os
warnings.filterwarnings('ignore')

# Set working directory to project root
# This notebook should be run from the project root, not from the notebooks folder
if Path.cwd().name == 'notebooks':
    os.chdir('..')
    print(f"Changed working directory to: {Path.cwd()}")
elif not (Path.cwd() / 'data' / 'processed').exists():
    print("⚠️ WARNING: Cannot find data/processed directory!")
    print(f"Current directory: {Path.cwd()}")
    print("Please run this notebook from the project root directory.")
else:
    print(f"Working directory: {Path.cwd()}")

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Paths
DATA_DIR = Path('data/processed')
RESULTS_DIR = Path('results')

# Verify paths exist
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR.absolute()}\nPlease ensure you're in the project root.")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("✓ Setup complete")

Changed working directory to: c:\Users\hrm01\drug_response_prediction
Device: cpu
✓ Setup complete


## 1. Load Data and Models

In [ ]:
# Load data
df = pd.read_csv(DATA_DIR / 'merged.csv')
expr_pca = pd.read_csv(DATA_DIR / 'gdsc_expr_pca.csv')
chemberta_feats = np.load(DATA_DIR / 'chemberta_drug_feats.npz')

# Load splits
train_idx = np.load(DATA_DIR / 'splits_drug/train_idx.npy')
val_idx = np.load(DATA_DIR / 'splits_drug/val_idx.npy')
test_idx = np.load(DATA_DIR / 'splits_drug/test_idx.npy')

# Filter valid indices
train_idx = train_idx[train_idx < len(df)]
val_idx = val_idx[val_idx < len(df)]
test_idx = test_idx[test_idx < len(df)]

# Detect column names (case-insensitive)
def find_column(df, possible_names):
    """Find column by trying different name variations"""
    cols_lower = {col.lower(): col for col in df.columns}
    for name in possible_names:
        if name.lower() in cols_lower:
            return cols_lower[name.lower()]
    return None

# Find correct column names with expanded search
cell_col = find_column(df, [
    'cell_id', 'CELL_ID',  # Added this!
    'cell_line_name', 'CELL_LINE_NAME', 
    'cellline', 'CELLLINE', 
    'cell_line', 'CELL_LINE',
    'cosmic_id', 'COSMIC_ID',
    'sanger_model_id', 'SANGER_MODEL_ID',
    'cell', 'CELL'
])
drug_col = find_column(df, ['drug_id', 'DRUG_ID'])
drug_name_col = find_column(df, ['drug_name', 'DRUG_NAME'])
ic50_col = find_column(df, ['ln_ic50', 'LN_IC50'])
tissue_col = find_column(df, ['tissue', 'TISSUE', 'tissue_type', 'TISSUE_TYPE'])

print(f"✓ Column mapping:")
print(f"  Cell line: {cell_col}")
print(f"  Drug ID: {drug_col}")
print(f"  Drug name: {drug_name_col}")
print(f"  IC50: {ic50_col}")
print(f"  Tissue: {tissue_col}")

# Check which are missing
missing = []
if not cell_col: missing.append("cell_line")
if not drug_col: missing.append("drug_id")
if not drug_name_col: missing.append("drug_name")
if not ic50_col: missing.append("ln_ic50")
if not tissue_col: missing.append("tissue")

if missing:
    print(f"\n❌ Missing columns: {', '.join(missing)}")
    print("Available columns:", df.columns.tolist()[:20])
    raise KeyError(f"Missing required columns: {', '.join(missing)}")

print(f"\nDataset: {len(df):,} samples")
print(f"Train: {len(train_idx):,}, Val: {len(val_idx):,}, Test: {len(test_idx):,}")
print(f"Drugs: {df[drug_col].nunique()}, Cell lines: {df[cell_col].nunique()}")

Available columns in merged.csv:
['cell_id', 'drug_id', 'drug_name', 'smiles', 'tissue', 'dataset', 'ln_ic50', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'PC11', 'PC12', 'PC13', 'PC14', 'PC15', 'PC16', 'PC17', 'PC18', 'PC19', 'PC20', 'PC21', 'PC22', 'PC23', 'PC24', 'PC25', 'PC26', 'PC27', 'PC28', 'PC29', 'PC30', 'PC31', 'PC32', 'PC33', 'PC34', 'PC35', 'PC36', 'PC37', 'PC38', 'PC39', 'PC40', 'PC41', 'PC42', 'PC43', 'PC44', 'PC45', 'PC46', 'PC47', 'PC48', 'PC49', 'PC50', 'PC51', 'PC52', 'PC53', 'PC54', 'PC55', 'PC56', 'PC57', 'PC58', 'PC59', 'PC60', 'PC61', 'PC62', 'PC63', 'PC64', 'PC65', 'PC66', 'PC67', 'PC68', 'PC69', 'PC70', 'PC71', 'PC72', 'PC73', 'PC74', 'PC75', 'PC76', 'PC77', 'PC78', 'PC79', 'PC80', 'PC81', 'PC82', 'PC83', 'PC84', 'PC85', 'PC86', 'PC87', 'PC88', 'PC89', 'PC90', 'PC91', 'PC92', 'PC93', 'PC94', 'PC95', 'PC96', 'PC97', 'PC98', 'PC99', 'PC100', 'PC101', 'PC102', 'PC103', 'PC104', 'PC105', 'PC106', 'PC107', 'PC108', 'PC109', 'PC110', 'PC111'

KeyError: 'Missing required columns: cell_line'

In [ ]:
# Load trained models
import sys
sys.path.append('.')
from src.models.mlp import MLP

# Get test data
df_test = df.iloc[test_idx].reset_index(drop=True)

# Load expression features
pc_cols = [col for col in df.columns if col.startswith('PC')]
X_expr_test = df_test[pc_cols].values

# Load tissue features
from sklearn.preprocessing import OneHotEncoder
ohe_tissue = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_tissue.fit(df[[tissue_col]])
X_tissue_test = ohe_tissue.transform(df_test[[tissue_col]])

# ChemBERTa features
chemberta_key = chemberta_feats.files[0]
if 'drug_id' in chemberta_feats.files:
    drug_ids_test = df_test[drug_col].values
    drug_id_to_idx = {did: i for i, did in enumerate(chemberta_feats['drug_id'])}
    drug_indices = [drug_id_to_idx[did] for did in drug_ids_test]
    X_drug_chemberta_test = chemberta_feats[chemberta_key][drug_indices]
else:
    X_drug_chemberta_test = chemberta_feats[chemberta_key][test_idx]

y_test = df_test[ic50_col].values

print(f"Expression: {X_expr_test.shape}")
print(f"ChemBERTa: {X_drug_chemberta_test.shape}")
print(f"Tissue: {X_tissue_test.shape}")
print(f"Using columns: cell={cell_col}, drug={drug_col}, tissue={tissue_col}, ic50={ic50_col}")

In [ ]:
# Load MLP+ChemBERTa model
checkpoint = torch.load(RESULTS_DIR / 'mlp_chemberta/mlp_model.pt', map_location=device)
in_dim = checkpoint['net.0.weight'].shape[1]

mlp_chemberta = MLP(
    in_dim=in_dim,
    hidden=[1024, 512],
    dropout=0.3
).to(device)

mlp_chemberta.load_state_dict(checkpoint)
mlp_chemberta.eval()

print(f"✓ Model loaded (expects {in_dim} features)")

In [ ]:
# Get model predictions
X_combined = np.column_stack([X_expr_test, X_drug_chemberta_test, X_tissue_test])

# Pad if needed
if X_combined.shape[1] < in_dim:
    padding = in_dim - X_combined.shape[1]
    X_combined = np.pad(X_combined, ((0, 0), (0, padding)), constant_values=0)
    print(f"Padded {padding} features")

with torch.no_grad():
    X_tensor = torch.FloatTensor(X_combined).to(device)
    preds = mlp_chemberta(X_tensor).cpu().numpy()

print(f"✓ Generated {len(preds)} predictions")

## 2. Critical Test: Do Predictions Vary by Cell Line?

**If models ignore cell features, predictions should be constant for each drug across all cell lines.**

In [ ]:
print("="*80)
print("TEST 1: Prediction Variance by Cell Line")
print("="*80)

# For each drug, check variance of predictions across cell lines
drug_variance = []

for drug_id in df_test[drug_col].unique():
    drug_mask = df_test[drug_col] == drug_id
    n_cell_lines = drug_mask.sum()
    
    if n_cell_lines > 1:  # Need multiple cell lines
        drug_preds = preds[drug_mask]
        drug_true = df_test[drug_mask][ic50_col].values
        
        drug_variance.append({
            'drug_id': drug_id,
            'drug_name': df_test[drug_mask][drug_name_col].iloc[0],
            'n_cell_lines': n_cell_lines,
            'pred_std': drug_preds.std(),
            'pred_range': drug_preds.max() - drug_preds.min(),
            'true_std': drug_true.std(),
            'true_range': drug_true.max() - drug_true.min()
        })

df_variance = pd.DataFrame(drug_variance)

print(f"\nAnalyzed {len(df_variance)} drugs tested on multiple cell lines")
print(f"\nPrediction variance statistics:")
print(f"  Mean Std Dev: {df_variance['pred_std'].mean():.4f}")
print(f"  Median Std Dev: {df_variance['pred_std'].median():.4f}")
print(f"  Mean Range: {df_variance['pred_range'].mean():.4f}")

print(f"\nTrue IC50 variance statistics:")
print(f"  Mean Std Dev: {df_variance['true_std'].mean():.4f}")
print(f"  Median Std Dev: {df_variance['true_std'].median():.4f}")
print(f"  Mean Range: {df_variance['true_range'].mean():.4f}")

# Verdict
print("\n" + "="*80)
if df_variance['pred_std'].mean() < 0.1:
    print("🚨 CRITICAL FINDING: Predictions barely vary by cell line!")
    print("   Model is predicting nearly constant values per drug.")
    print("   This confirms models IGNORE gene expression and tissue features.")
elif df_variance['pred_std'].mean() < df_variance['true_std'].mean() * 0.3:
    print("⚠️  WARNING: Predictions vary much less than true values.")
    print("   Model under-utilizes cell line information.")
else:
    print("✓ Predictions vary appropriately across cell lines.")
    print("   Model appears to use cell line features.")
print("="*80)

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of prediction std devs
axes[0].hist(df_variance['pred_std'], bins=30, alpha=0.7, label='Predictions', color='steelblue')
axes[0].hist(df_variance['true_std'], bins=30, alpha=0.7, label='True Values', color='coral')
axes[0].set_xlabel('Std Dev Across Cell Lines', fontsize=11)
axes[0].set_ylabel('Number of Drugs', fontsize=11)
axes[0].set_title('Distribution of Variance Across Cell Lines', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Scatter: predicted vs true variance
axes[1].scatter(df_variance['true_std'], df_variance['pred_std'], alpha=0.5, s=30)
axes[1].plot([0, df_variance['true_std'].max()], [0, df_variance['true_std'].max()], 
             'r--', linewidth=2, label='Perfect Correlation')
axes[1].set_xlabel('True IC50 Std Dev', fontsize=11)
axes[1].set_ylabel('Predicted IC50 Std Dev', fontsize=11)
axes[1].set_title('Predicted vs True Variance', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Interpretation:")
print("   • Left: If prediction variance << true variance → model ignores cell context")
print("   • Right: Points should cluster near diagonal for good personalization")

In [ ]:
# Show specific examples
print("\nExamples - Drugs with high true variance but low predicted variance:")
print("(These drugs SHOULD have personalized predictions but DON'T)\n")

df_variance['variance_gap'] = df_variance['true_std'] - df_variance['pred_std']
worst_drugs = df_variance.nlargest(10, 'variance_gap')

print(worst_drugs[['drug_name', 'n_cell_lines', 'pred_std', 'true_std', 'variance_gap']].to_string(index=False))

## 3. Data Leakage Check

**Check if same drugs appear in both train and test sets.**

In [ ]:
print("="*80)
print("TEST 2: Data Leakage Analysis")
print("="*80)

# Get unique drugs in each split
drugs_train = set(df.iloc[train_idx][drug_col].unique())
drugs_val = set(df.iloc[val_idx][drug_col].unique())
drugs_test = set(df.iloc[test_idx][drug_col].unique())

print(f"\nUnique drugs per split:")
print(f"  Train: {len(drugs_train)}")
print(f"  Val: {len(drugs_val)}")
print(f"  Test: {len(drugs_test)}")
print(f"  Total unique: {len(drugs_train | drugs_val | drugs_test)}")

# Check overlaps
overlap_train_test = drugs_train & drugs_test
overlap_train_val = drugs_train & drugs_val
overlap_val_test = drugs_val & drugs_test

print(f"\nOverlap analysis:")
print(f"  Train ∩ Test: {len(overlap_train_test)} drugs ({len(overlap_train_test)/len(drugs_test)*100:.1f}% of test)")
print(f"  Train ∩ Val: {len(overlap_train_val)} drugs ({len(overlap_train_val)/len(drugs_val)*100:.1f}% of val)")
print(f"  Val ∩ Test: {len(overlap_val_test)} drugs ({len(overlap_val_test)/len(drugs_test)*100:.1f}% of test)")

# Verdict
print("\n" + "="*80)
if len(overlap_train_test) > 0:
    print("🚨 DATA LEAKAGE DETECTED!")
    print(f"   {len(overlap_train_test)} drugs appear in BOTH train and test sets.")
    print("   Models can memorize average drug potency from training.")
    print("   This explains why drug features dominate predictions!")
else:
    print("✓ NO LEAKAGE: Train and test have completely different drugs.")
    print("   This is a proper 'drug split' for testing generalization.")
    print("   Models cannot memorize drugs - must learn from chemical structure.")
print("="*80)

## 4. Signal Strength Analysis

**Compare variance explained by drugs vs. cell lines in the ground truth.**

In [ ]:
print("="*80)
print("TEST 3: Signal Strength - Drug vs Cell Line")
print("="*80)

# Compute variance decomposition
from scipy.stats import f_oneway

# Group IC50 by drug and by cell line
grouped_by_drug = df_test.groupby(drug_col)[ic50_col].apply(list)
grouped_by_cell = df_test.groupby(cell_col)[ic50_col].apply(list)

# ANOVA F-statistic (larger = more variance explained)
if len(grouped_by_drug) > 1:
    f_stat_drug, p_drug = f_oneway(*[g for g in grouped_by_drug if len(g) > 0])
else:
    f_stat_drug, p_drug = 0, 1

if len(grouped_by_cell) > 1:
    f_stat_cell, p_cell = f_oneway(*[g for g in grouped_by_cell if len(g) > 0])
else:
    f_stat_cell, p_cell = 0, 1

print(f"\nANOVA F-statistics (higher = more variance explained):")
print(f"  By Drug: F={f_stat_drug:.2f}, p={p_drug:.2e}")
print(f"  By Cell Line: F={f_stat_cell:.2f}, p={p_cell:.2e}")

# Compute mean std within groups
drug_within_std = df_test.groupby(drug_col)[ic50_col].std().mean()
cell_within_std = df_test.groupby(cell_col)[ic50_col].std().mean()
total_std = df_test[ic50_col].std()

print(f"\nStandard deviation:")
print(f"  Total (all data): {total_std:.4f}")
print(f"  Within drugs (across cell lines): {drug_within_std:.4f}")
print(f"  Within cell lines (across drugs): {cell_within_std:.4f}")

# Interpretation
print("\n" + "="*80)
if f_stat_drug > f_stat_cell * 3:
    print("🔬 SCIENTIFIC FINDING: Drug identity explains FAR more variance than cell line.")
    print("   This may justify why models focus on drug features!")
    print("   Cell line variation might be noise relative to drug effects.")
elif f_stat_cell > f_stat_drug * 3:
    print("⚠️  Cell lines explain more variance - models SHOULD use cell features!")
    print("   Current models are missing important personalization signal.")
else:
    print("📊 Both drugs and cell lines contribute substantially to variance.")
    print("   Ideal model should integrate both types of features.")
print("="*80)

In [ ]:
# Visualize signal sources
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot: IC50 distribution by top drugs
top_drugs = df_test[drug_name_col].value_counts().head(10).index
df_top_drugs = df_test[df_test[drug_name_col].isin(top_drugs)]

axes[0].violinplot(
    [df_top_drugs[df_top_drugs[drug_name_col]==d][ic50_col].values for d in top_drugs],
    positions=range(len(top_drugs)),
    showmeans=True,
    widths=0.7
)
axes[0].set_xticks(range(len(top_drugs)))
axes[0].set_xticklabels(top_drugs, rotation=45, ha='right')
axes[0].set_ylabel('LN_IC50', fontsize=11)
axes[0].set_title('IC50 Distribution by Drug', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3, axis='y')

# Box plot: IC50 distribution by top cell lines
top_cells = df_test[cell_col].value_counts().head(10).index
df_top_cells = df_test[df_test[cell_col].isin(top_cells)]

axes[1].violinplot(
    [df_top_cells[df_top_cells[cell_col]==c][ic50_col].values for c in top_cells],
    positions=range(len(top_cells)),
    showmeans=True,
    widths=0.7
)
axes[1].set_xticks(range(len(top_cells)))
axes[1].set_xticklabels(top_cells, rotation=45, ha='right')
axes[1].set_ylabel('LN_IC50', fontsize=11)
axes[1].set_title('IC50 Distribution by Cell Line', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n💡 Interpretation:")
print("   • Wider violins = more variance within that group")
print("   • If drugs show narrow violins → consistent across cell lines (drug-only signal)")
print("   • If cell lines show narrow violins → consistent across drugs (cell-only signal)")

## 5. Feature Magnitude Check

**Check if expression features have much smaller magnitude than drug features (scaling issue).**

In [ ]:
print("="*80)
print("TEST 4: Feature Scale Analysis")
print("="*80)

print(f"\nFeature magnitudes:")
print(f"  Expression (PCs): mean={X_expr_test.mean():.6f}, std={X_expr_test.std():.6f}, range=[{X_expr_test.min():.2f}, {X_expr_test.max():.2f}]")
print(f"  Drug (ChemBERTa): mean={X_drug_chemberta_test.mean():.6f}, std={X_drug_chemberta_test.std():.6f}, range=[{X_drug_chemberta_test.min():.2f}, {X_drug_chemberta_test.max():.2f}]")
print(f"  Tissue (one-hot): mean={X_tissue_test.mean():.6f}, std={X_tissue_test.std():.6f}, range=[{X_tissue_test.min():.2f}, {X_tissue_test.max():.2f}]")

# Check if features need standardization
expr_scale = X_expr_test.std()
drug_scale = X_drug_chemberta_test.std()
scale_ratio = drug_scale / expr_scale if expr_scale > 0 else 0

print(f"\nScale ratio (Drug/Expression): {scale_ratio:.2f}x")

print("\n" + "="*80)
if scale_ratio > 5:
    print("⚠️  Drug features have much larger magnitude than expression!")
    print("   This can cause models to ignore smaller-scale features.")
    print("   Consider standardizing all features before training.")
elif scale_ratio < 0.2:
    print("⚠️  Expression features have much larger magnitude than drugs!")
    print("   Consider standardizing all features before training.")
else:
    print("✓ Features are reasonably scaled relative to each other.")
    print("   Scale mismatch is not the issue.")
print("="*80)

## 6. Summary Report

**Consolidated findings and recommendations.**

In [ ]:
print("\n" + "="*90)
print(" "*30 + "FINAL DIAGNOSIS")
print("="*90)

# Collect test results
pred_variance_low = df_variance['pred_std'].mean() < 0.1
has_leakage = len(overlap_train_test) > 0
drug_dominant = f_stat_drug > f_stat_cell * 3
scale_issue = scale_ratio > 5 or scale_ratio < 0.2

print("\n📋 TEST RESULTS:")
print(f"  1. Predictions vary by cell line: {'❌ NO' if pred_variance_low else '✅ YES'}")
print(f"  2. Data leakage present: {'🚨 YES' if has_leakage else '✅ NO'}")
print(f"  3. Drug signal dominates: {'⚠️ YES' if drug_dominant else '✅ BALANCED'}")
print(f"  4. Feature scaling issue: {'⚠️ YES' if scale_issue else '✅ NO'}")

print("\n🔍 ROOT CAUSE ANALYSIS:")

if has_leakage:
    print("\n  PRIMARY ISSUE: Data Leakage 🚨")
    print("  → Same drugs in train and test sets")
    print("  → Models memorized average drug potency")
    print("  → Performance metrics are inflated")
    print("\n  RECOMMENDATION:")
    print("  • Results are invalid for new drug discovery")
    print("  • If goal is drug-specific models: this is fine (but not personalized)")
    print("  • If goal is new drug generalization: splits are correct, leakage is expected")

elif pred_variance_low and drug_dominant:
    print("\n  PRIMARY ISSUE: Drug Signal Overwhelms Cell Signal 📊")
    print("  → Drug identity explains most variance in ground truth")
    print("  → Cell line variation is smaller and possibly noisy")
    print("  → Models rationally focus on dominant signal")
    print("\n  RECOMMENDATION:")
    print("  • This may be scientifically correct!")
    print("  • Consider: Is personalization valuable for this dataset?")
    print("  • Try interaction-based architecture to force cell feature usage")
    print("  • Validate: Check high cell-variance drugs specifically")

elif pred_variance_low and scale_issue:
    print("\n  PRIMARY ISSUE: Feature Scaling Problem ⚖️")
    print("  → Expression features have different magnitude than drug features")
    print("  → Neural network gradients favor larger-scale features")
    print("\n  RECOMMENDATION:")
    print("  • Standardize ALL features before training (mean=0, std=1)")
    print("  • Retrain with: StandardScaler on each feature group")
    print("  • This is an easy fix!")

elif pred_variance_low:
    print("\n  PRIMARY ISSUE: Model Architecture Limitation 🏗️")
    print("  → Simple MLP with concatenation doesn't force feature interaction")
    print("  → Model found shortcut: drug features are sufficient")
    print("\n  RECOMMENDATION:")
    print("  • Use interaction layers: drug_feats * cell_feats")
    print("  • Try attention mechanism: drug attends to expression")
    print("  • Multi-head: separate encoders + fusion layer")
    print("  • Add regularization that penalizes ignoring feature groups")

else:
    print("\n  ✅ Models appear to use cell line features appropriately!")
    print("  → SHAP analysis may have sampling issues (too few background samples)")
    print("  → Try increasing SHAP samples or use DeepExplainer")

print("\n" + "="*90)
print("\n💡 NEXT STEPS:")
print("  1. Review test results above to understand your specific situation")
print("  2. If drug signal dominates: Consider if personalization is feasible")
print("  3. If architecture issue: Retrain with interaction-based model")
print("  4. If scale issue: Retrain with standardized features")
print("  5. If leakage: Clarify evaluation goals (personalization vs. new drugs)")
print("="*90)

## 7. Optional: Quick Ablation Test

**Train a drug-only model to confirm expression/tissue features are unused.**

If drug-only model achieves similar performance, confirms the finding.

In [ ]:
# Train simple drug-only baseline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

print("="*80)
print("ABLATION TEST: Drug-Only vs Full Model")
print("="*80)

# Prepare train/test data
X_drug_train = chemberta_feats[chemberta_key][train_idx] if 'drug_id' not in chemberta_feats.files else None
y_train = df.iloc[train_idx][ic50_col].values

if X_drug_train is not None:
    # Train drug-only model
    print("\nTraining Ridge regression with drug features only...")
    model_drug_only = Ridge(alpha=1.0)
    model_drug_only.fit(X_drug_train, y_train)
    
    # Predictions
    preds_drug_only = model_drug_only.predict(X_drug_chemberta_test)
    
    # Metrics
    mse_drug_only = mean_squared_error(y_test, preds_drug_only)
    r2_drug_only = r2_score(y_test, preds_drug_only)
    
    mse_full = mean_squared_error(y_test, preds)
    r2_full = r2_score(y_test, preds)
    
    print(f"\nPerformance Comparison:")
    print(f"  Drug-Only Model: MSE={mse_drug_only:.4f}, R²={r2_drug_only:.4f}")
    print(f"  Full Model:      MSE={mse_full:.4f}, R²={r2_full:.4f}")
    print(f"  Difference:      ΔMSE={mse_full-mse_drug_only:.4f}, ΔR²={r2_full-r2_drug_only:.4f}")
    
    print("\n" + "="*80)
    if abs(r2_full - r2_drug_only) < 0.05:
        print("🚨 CONFIRMED: Drug-only model performs nearly as well!")
        print("   Expression and tissue features add minimal value.")
        print("   Full model is essentially learning drug-only predictions.")
    else:
        print("✅ Full model outperforms drug-only significantly.")
        print("   Cell line features DO contribute to performance.")
    print("="*80)
else:
    print("\n⚠️  Skipping ablation - need to reconstruct ChemBERTa features for train set.")
    print("   Run this in your training script to get definitive answer.")